In [1]:
# !pip install -U \
#     "transformers>=4.56.2,<5" \
#     "trl[peft]" \
#     "peft>=0.19.1" \
#     "datasets" \
#     "accelerate" \
#     "bitsandbytes"

In [2]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)
from trl import (
    SFTTrainer,
    SFTConfig,
)

W0815 00:07:54.411000 31544 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W0815 00:07:55.308000 31544 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [11]:
MODEL_NAME = "meta-llama/Llama-3.2-3B"

OUTPUT_DIR = "./llama-3.2-3b-sft"

# Start small for testing.
MAX_TRAIN_SAMPLES = 15_000

MAX_SEQ_LENGTH = 768

SEED = 42


In [12]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. This training script expects an NVIDIA GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")



GPU: NVIDIA GeForce RTX 4070
VRAM: 11.99 GB


In [13]:

print("\nLoading tokenizer...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# Llama doesn't normally have a padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.chat_template = """{% for message in messages %}
{% if message['role'] == 'user' %}
<|start_header_id|>user<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% elif message['role'] == 'assistant' %}
<|start_header_id|>assistant<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% elif message['role'] == 'system' %}
<|start_header_id|>system<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
<|start_header_id|>assistant<|end_header_id|>

{% endif %}"""


Loading tokenizer...


In [14]:
print("\nLoading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    # Use BF16 on GPUs that support it.
    torch_dtype=torch.bfloat16,

    device_map={"": 0},

    # Saves some GPU memory during training.
    use_cache=False,
)

model.config.pad_token_id = tokenizer.pad_token_id


model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()


Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
print("\nLoading UltraChat...")

dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
)

dataset = dataset.shuffle(seed=SEED)

dataset = dataset.select(
    range(min(MAX_TRAIN_SAMPLES, len(dataset)))
)


def convert_to_prompt_completion(example):
    messages = example["messages"]

    # Find the final assistant message
    completion = messages[-1]

    # Everything before the final assistant response
    prompt = messages[:-1]

    return {
        "prompt": prompt,
        "completion": [completion],
    }


dataset = dataset.map(
    convert_to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=2,
)

print("Training examples:", len(dataset))
print(dataset[0])


Loading UltraChat...


Map (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

Training examples: 15000
{'prompt': [{'content': 'How does the location of the Sydney Conservatorium of Music impact the academic and professional opportunities available to music students, and how does the conservatorium support student engagement with the music industry in Australia?', 'role': 'user'}, {'content': "The location of the Sydney Conservatorium of Music, which is situated in the heart of Sydney's cultural precinct, impacts both the academic and professional opportunities available to music students. The conservatorium is located near several major performing arts venues and organizations, including the Sydney Opera House, the Australian Broadcasting Corporation, and the Sydney Symphony Orchestra, providing students with easy access to performances, rehearsals, and networking opportunities.\n\nOne of the primary ways the conservatorium supports student engagement with the music industry in Australia is through its strong industry connections. The conservatorium has establi

In [16]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [17]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=1,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    # Sequence length
    max_length=MAX_SEQ_LENGTH,

    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Save checkpoints
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    # Evaluation is omitted here because we're doing a first
    # training run without a validation split.

    # Precision
    fp16=False,
    bf16=True,

    # Memory/performance
    gradient_checkpointing=True,

    # Optimizer
    optim="paged_adamw_8bit",

    # Dataset processing
    dataset_num_proc=2,

    # Reproducibility
    seed=SEED,

    # Report to nothing by default
    report_to="none",

    # Don't remove dataset columns automatically.
    remove_unused_columns=False,

    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=dataset,

    processing_class=tokenizer,

    peft_config=lora_config,
)

print("\nStarting training...\n")

trainer.train()


Tokenizing train dataset (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

Building labels for train dataset (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset (num_proc=2):   0%|          | 0/15000 [00:00<?, ? examples/…

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.



Starting training...



Step,Training Loss
1,0.917200
10,1.180700
20,1.186800
30,1.003800
40,1.070600
50,1.050000
60,1.093600
70,1.062600
80,1.061000
90,1.032300


TrainOutput(global_step=858, training_loss=1.007026008445344, metrics={'train_runtime': 4642.9941, 'train_samples_per_second': 1.477, 'train_steps_per_second': 0.185, 'total_flos': 7.621695652005888e+16, 'train_loss': 1.007026008445344, 'entropy': 0.9955867768849357, 'num_tokens': 4468010.0, 'mean_token_accuracy': 0.7319118390649052, 'epoch': 1.0})

In [18]:
print("\nSaving model...")

trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining complete!")
print("LoRA adapter saved to:", OUTPUT_DIR)


Saving model...

Training complete!
LoRA adapter saved to: ./llama-3.2-3b-sft
